# We implement the rules following Lee and Seung 2001, and Morup and Hansen 2015

In [1]:
from tensorly_custom.tucker_tensor import validate_tucker_rank, tucker_normalize, TuckerTensor
from tensorly_custom.decomposition._tucker import tucker_to_tensor
from tensorly_custom.base import unfold
from tensorly_custom.tenalg import mode_dot
from datetime import datetime
import tensorly_custom as tl
import numpy as np
import torch
import os
from typing import List, Tuple
from utils import DATA_DIR, select_gpu, tree_to_device, notify_discord
from typing import Optional, Union
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import time
import json

from tucker_tensor import SparseTupleTensor

device = select_gpu()
tl.set_backend("cupy")

# ----- cupy sparse methods -----

def fro_norm_coo(x):
    # x: cupyx.scipy.sparse.coo_matrix
    data = x.data
    return cp.sqrt((cp.abs(data) ** 2).sum())

def unfold_from_vectorized_sparse(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape,
    mode: int,
    to_dense: bool = False,
):
    """
    Unfold a sparse tensor that is stored as a vectorized CuPy sparse matrix.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.spmatrix
        Sparse matrix of shape (np.prod(orig_shape), 1) created by
        `torch_sparse_to_cupy` for an N-D tensor.
    orig_shape : tuple[int, ...]
        Original N-D tensor shape, e.g. (I0, I1, I2).
    mode : int
        Mode along which to unfold.
    to_dense : bool, default False
        If True, return a dense cupy.ndarray.
        If False, return a cupy sparse COO matrix.

    Returns
    -------
    unfolded : cupy.ndarray or cupyx.scipy.sparse.coo_matrix
        Mode-`mode` unfolding of shape
        (orig_shape[mode], np.prod(orig_shape) // orig_shape[mode]).
    """
    # Make sure we're in COO format

    cu = vec_tensor.tocoo()

    row_cp = cu.row
    col_cp = cu.col
    data_cp = cu.data

    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # ---- move to host and use int64 for safe arithmetic ----
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)

    coords = np.unravel_index(flat_np, orig_shape)

    row_unf_np = coords[mode]

    other_coords = coords[:mode] + coords[mode + 1:]
    other_shape = tuple(s for i, s in enumerate(orig_shape) if i != mode)
    col_unf_np = np.ravel_multi_index(other_coords, other_shape)

    row_unf_cp = cp.asarray(row_unf_np)
    col_unf_cp = cp.asarray(col_unf_np)

    unfolded_shape = (orig_shape[mode], int(np.prod(other_shape)))
    unfolded = cpx_sparse.coo_matrix(
        (data_cp, (row_unf_cp, col_unf_cp)),
        shape=unfolded_shape,
    )

    if to_dense:
        return unfolded.toarray()
    return unfolded


def left_dense_mul_sparse(
    mat: cp.ndarray,
    sp: cpx_sparse.spmatrix
) -> Union[cp.ndarray, cpx_sparse.coo_matrix]:
    """
    Compute mat @ sp, choosing dense or sparse output based on a simple
    memory heuristic.

    mat: cupy ndarray of shape (R, I_mode)
    sp:  cupy sparse matrix of shape (I_mode, K)
    """
    sp = sp.tocoo()
    R, I_mode = mat.shape
    assert I_mode == sp.shape[0], f"mat shape {mat.shape} not compatible with sparse {sp.shape}"

    # Let CuPy handle dense @ sparse; result is cupy.ndarray
    return mat @ sp
def fold_unfolded_sparse_to_vec(
    unfolded: Union[cpx_sparse.spmatrix, cp.ndarray],
    old_shape: Tuple[int, ...],
    mode: int,
    new_dim: int,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Fold a mode-`mode` unfolded matrix back to a vectorized sparse tensor.

    unfolded:
        - sparse COO or any cupyx.scipy.sparse.spmatrix of shape (new_dim, prod(other_dims)), or
        - dense cupy.ndarray of the same shape.
    old_shape : original N-D shape BEFORE replacing dimension at `mode`
    mode      : mode index that was unfolded
    new_dim   : new size at `mode` (typically rank[mode])

    Returns
    -------
    vec_sparse : COO of shape (prod(new_shape), 1)
    new_shape  : tuple of ints, updated tensor shape
    """

    old_shape = tuple(old_shape)
    N = len(old_shape)

    new_shape = list(old_shape)
    new_shape[mode] = new_dim
    new_shape = tuple(new_shape)

    other_shape = tuple(s for i, s in enumerate(old_shape) if i != mode)

    if cpx_sparse.isspmatrix(unfolded):
        unfolded = unfolded.tocoo()
        row = unfolded.row
        col = unfolded.col
        data = unfolded.data
    else:
        row, col = cp.nonzero(unfolded)
        data = unfolded[row, col]

    coords_other = cp.unravel_index(col, other_shape)

    coords_full = []
    idx_other = 0
    for i in range(N):
        if i == mode:
            coords_full.append(row)
        else:
            coords_full.append(coords_other[idx_other])
            idx_other += 1

    coords_full = tuple(coords_full)

    size = int(np.prod(new_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    flat = cp.ravel_multi_index(coords_full, new_shape)

    # --- block encoding of flat indices ---
    row_vec = flat % block_size
    col_vec = flat // block_size

    n_blocks = int((size + block_size - 1) // block_size)
    vec_sparse = cpx_sparse.coo_matrix(
        (data, (row_vec, col_vec)),
        shape=(block_size, n_blocks),
    )
    vec_sparse.sum_duplicates()

    return vec_sparse, new_shape


def sparse_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    curr_shape: Tuple[int, ...],
    factor: cp.ndarray,
    mode: int,
    transpose_factor: bool = True,
) -> Tuple[cpx_sparse.coo_matrix, Tuple[int, ...]]:
    """
    Perform a mode-`mode` product on a vectorized sparse tensor (prod(curr_shape), 1),
    using a dense factor matrix, and return the new vectorized sparse tensor.

    vec_tensor: sparse COO (prod(curr_shape), 1)
    curr_shape: current tensor shape
    factor:     dense matrix of shape (I_mode, R_mode) (or R_mode, I_mode if transpose_factor=False)
    mode:       mode index in [0, len(curr_shape))
    transpose_factor: if True, use factor.T (for Tucker-style X ×_n W_n^T)

    Returns
    -------
    new_vec:   sparse COO (prod(new_shape), 1)
    new_shape: updated shape, with dimension at `mode` replaced by R_mode
    """
    curr_shape = tuple(curr_shape)
    I_mode = curr_shape[mode]

    # Factor handling
    if transpose_factor:
        # factor is (I_mode, R_mode) => mat is (R_mode, I_mode)
        assert factor.shape[0] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = tl.transpose(factor)  # (R_mode, I_mode)
    else:
        # factor is already (R_mode, I_mode)
        assert factor.shape[1] == I_mode, f"factor shape {factor.shape} not compatible with dim {I_mode}"
        mat = factor

    R_mode = mat.shape[0]

    # 1) Unfold current sparse tensor along this mode (sparse COO)
    unfolded = unfold_from_vectorized_sparse(
        vec_tensor,
        curr_shape,
        mode,
        to_dense=False,
    )  # shape: (I_mode, prod(other_dims))

    # 2) Left-multiply with dense matrix; currently returns dense cp.ndarray
    #    -> shape: (R_mode, prod(other_dims))
    unfolded_new = left_dense_mul_sparse(mat, unfolded)

    # 3) Fold back into a new vectorized sparse tensor with updated shape
    new_vec, new_shape = fold_unfolded_sparse_to_vec(
        unfolded_new,
        old_shape=curr_shape,
        mode=mode,
        new_dim=R_mode,
    )
    return new_vec, new_shape
def sparse_multi_mode_dot_vec(
    vec_tensor: cpx_sparse.spmatrix,
    orig_shape: Tuple[int, ...],
    factors: List[cp.ndarray],
    modes: Optional[List[int]] = None,
    transpose_factors: bool = True,
) -> cp.ndarray:
    """
    multi_mode_dot for a vectorized sparse tensor (prod(orig_shape), 1),
    applying dense factor matrices along the given modes, **staying sparse**
    until the final (small) result, which is densified.

    vec_tensor: sparse COO (prod(orig_shape), 1)
    orig_shape: original tensor shape
    factors:    list of factor matrices, one per mode index
                factor[n] has shape (I_n, R_n)
    modes:      list of modes to apply; if None, uses range(len(factors))
    transpose_factors: if True, uses factors[n].T (Tucker-style)
    """
    if modes is None:
        modes = list(range(len(factors)))

    current_vec = vec_tensor
    current_shape = tuple(orig_shape)

    # Apply each mode in any order (commutes)
    for mode in modes:
        # print("Applying mode", mode)
        current_vec, current_shape = sparse_mode_dot_vec(
            current_vec,
            current_shape,
            factors[mode],
            mode=mode,
            transpose_factor=transpose_factors,
        )

    # At this point, current_vec is still sparse (prod(core_shape), 1)
    core_shape = current_shape  # typically your (50, 50, 50) or similar
    # should not overflow the cupy 32bit index limit if dimensions stay reasonable
    # Finally densify the small core
    coo = current_vec.tocoo()
    flat = coo.row
    data = coo.data

    # Build dense core
    coords = cp.unravel_index(flat, core_shape)
    core_dense = cp.zeros(core_shape, dtype=data.dtype)
    core_dense[coords] = data

    return core_dense

def print_elapsed_time(start_time, message=""):
    """Prints the elapsed time since start_time."""
    now = time.time()
    # if the message starts with indents (tabs), add the same number to the elapsed time print
    tabs = ""
    for char in message:
        if char == "\t":
            tabs += "\t"
        else:
            break
    elapsed_time = now - start_time
    minutes, seconds = divmod(elapsed_time, 60)
    if message:
        print(message)
    print(f"{tabs}Elapsed time: {int(minutes)} minutes and {seconds} seconds.")
    return now


2025-12-12 11:05:52.963116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Selected GPU 3 with 454.62 MB used memory.


In [2]:
from tucker_tensor import SparseTupleTensor
import tensorly_custom as tl
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=1000,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("cupy")
n_iter_max=10
init="random"
tol=10e-5
random_state=42
verbose=False
return_errors="full"
normalize_factors=False
patience: int=3
rank = [50, 50, 50]

In [3]:
positive = tl.sum(sparse_tensor.tensor.data > 0).item()
negative = tl.sum(sparse_tensor.tensor.data < 0).item()
zeros = tl.sum(sparse_tensor.tensor.data == 0).item()
print(f"negative:{negative}, positive:{positive}, zeros:{zeros}")

negative:100744, positive:653363, zeros:0


In [4]:
shape = tuple(sparse_tensor.shape)
tensor = sparse_tensor.tensor
rank = validate_tucker_rank(shape, rank=rank)
epsilon = 1e-12
modes = list(range(len(rank)))
non_negative = True
# skip_factor = None
# transpose_factors = False


if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])), # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")


# Frobenius norm (Euclidian distance)
## factors

for each mode *n*:
- $X_n$ = mode-$n$ unfolding of $X$ (`unfolded`)
- $\mathcal{B}$ = $\mathcal{G} \times _{k\neq n} A^{k}$ (the core multiplied by the factors except for the current mode)
- $B$ = transposed version of $\mathcal{B}$


$$
A^{(n)} \leftarrow A^{(n)} \odot
\frac{X_{(n)} B}{A^{(n)} B^{\mathsf T} B}
$$

i.e. elementwise

$$
A^{(n)}_{i_n \lambda}
\leftarrow
A^{(n)}_{i_n \lambda}
,
\frac{ \bigl(X_{(n)} B\bigr)_{i_n \lambda} }
{ \bigl(A^{(n)} B^{\mathsf T} B\bigr)_{i_n \lambda} }.
$$


In [6]:
start = time.time()
test_factors = [f.copy() for f in factors]
for mode in modes:
    B = tucker_to_tensor((core, factors), skip_factor=mode)
    B = tl.transpose(unfold(B, mode))
    unfolded = unfold_from_vectorized_sparse(tensor, shape, mode)
    print(type(unfolded), unfolded.shape)
    print(type(B), B.shape)
    numerator   = unfolded @ B
    denominator = tl.dot(factors[mode], tl.dot(tl.transpose(B), B))
    test_factors[mode] *= numerator / denominator
factor_time_frob  = print_elapsed_time(start, "Factor updates done.")

<class 'cupyx.scipy.sparse._coo.coo_matrix'> (1000, 1000000)
<class 'cupy.ndarray'> (1000000, 50)
<class 'cupyx.scipy.sparse._coo.coo_matrix'> (1000, 1000000)
<class 'cupy.ndarray'> (1000000, 50)
<class 'cupyx.scipy.sparse._coo.coo_matrix'> (1000, 1000000)
<class 'cupy.ndarray'> (1000000, 50)
Factor updates done.
Elapsed time: 0 minutes and 0.12266802787780762 seconds.


Exactly following the Morup paper notation:
$$R_{(n)} = A^{(n)}Z_{(n)}$$
$$A^{(n)} \leftarrow A^{(n)}\cdot (\frac{X_{(n)}Z^{T}_{(n)}}{R_{(n)}Z^{T}_{(n)}})$$

In [6]:
for mode_n in modes:
    X = unfold_from_vectorized_sparse(tensor, shape, mode_n)
    Z = tucker_to_tensor((core, factors), skip_factor=mode_n)

    # Exactly like the formula, but inefficient due to A * Z intermediate shape blowing up
    # Z = unfold(Z, mode_n)
    # Z_T = tl.transpose(Z)
    # A = factors[mode_n]
    # R = tl.dot(A, Z)
    # numerator = X @ Z_T
    # denominator = tl.dot(R, Z_T)


    Z = tl.transpose(unfold(Z, mode_n))
    A = factors[mode_n]
    numerator = X @ Z
    print("Z shape", Z.shape)
    print("Z-Z^T dot shape:", tl.dot(tl.transpose(Z), Z).shape)
    print("A shape", A.shape)
    denominator = tl.dot(A, tl.dot(tl.transpose(Z), Z))
    factors[mode_n] *= numerator / denominator

Z shape (1000000, 50)
Z-Z^T dot shape: (50, 50)
A shape (1000, 50)
Z shape (1000000, 50)
Z-Z^T dot shape: (50, 50)
A shape (1000, 50)
Z shape (1000000, 50)
Z-Z^T dot shape: (50, 50)
A shape (1000, 50)


## core

- The **numerator** is the multi-mode projection of (X) with transposed factors
  $$
X \times_1 A^{(1)\mathsf T}
\times_2 A^{(2)\mathsf T}
\cdots
\times_N A^{(N)\mathsf T}.
$$

- The **denominator** is the multi-mode projection of the current core with current factor matrices
$$
\mathcal{G} \times_1 \bigl(A^{(1)\mathsf T} A^{(1)}\bigr)
\times_2 \bigl(A^{(2)\mathsf T} A^{(2)}\bigr)
\cdots
\times_N \bigl(A^{(N)\mathsf T} A^{(N)}\bigr).
$$

Then the core update is

$$
\mathcal{G} \leftarrow \mathcal{G} \odot \frac{\text{num}}{\text{den}}
$$


In [7]:
# we copy the core
core_copy = core.copy()

In [8]:
start = time.time()
numerator = sparse_multi_mode_dot_vec(
    vec_tensor=tensor,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)
numerator = tl.clip(numerator, a_min=epsilon, a_max=None)

for i, f in enumerate(factors):
    if i:
        denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
    else:
        denominator = mode_dot(core, tl.dot(tl.transpose(f), f), i)
denominator = tl.clip(denominator, a_min=epsilon, a_max=None)

core *= numerator / denominator
core_time_frob = print_elapsed_time(start, "Core update done.")

Core update done.
Elapsed time: 0 minutes and 0.4451143741607666 seconds.


Exactly as in Morup:

With: $\mathcal{R} = \mathcal{G} \Phi$

$\mathcal{B} = \mathcal{X} \Phi^T$

$\mathcal{C} = \mathcal{R} \Phi^T$

$$\mathcal{G} \leftarrow \mathcal{G} \cdot (\frac{\mathcal{B}}{\mathcal{C}})$$

In [9]:
start = time.time()

B = sparse_multi_mode_dot_vec(
    vec_tensor=tensor,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)
B = tl.clip(B, a_min=epsilon, a_max=None)
R = tucker_to_tensor((core_copy, factors))
for i, factor in enumerate(factors):
    R = mode_dot(R, tl.transpose(factor), i)
R = tl.clip(R, a_min=epsilon, a_max=None)

core_copy *= B / R
core_time_frob = print_elapsed_time(start, "Core update done.")


Core update done.
Elapsed time: 0 minutes and 0.2911968231201172 seconds.


In [10]:
# we check if core and core_copy are exactly the same (up to numerical tolerance)
atol = 1e-6
rtol = 1e-6

diff_found = False
for i, (slc_core, slc_copy) in enumerate(zip(core, core_copy)):
    if not np.allclose(slc_core, slc_copy, atol=atol, rtol=rtol):
        print(f"Mismatch in slice {i}")
        max_diff = tl.max(tl.abs(slc_core - slc_copy))
        print(f"Max abs diff in this slice: {max_diff}")
        # we print the mean of the slice
        mean = tl.mean(slc_core)
        mean_copy = tl.mean(slc_copy)
        print(f"Mean in the slices: {mean_copy} - {mean}")
        diff_found = True


if not diff_found:
    print("core and core_copy match within numerical tolerance.")


core and core_copy match within numerical tolerance.


## Reconstruction error
- $|X|_F^2$ = `norm_X_sq`

- $| \hat X |_F^2 = \langle \mathcal{G}, \text{den} \rangle$

- $\langle X, \hat X \rangle = \langle \mathcal{G}, \text{num} \rangle$


Then

$$
|X - \hat X|_F^2
= |X|_F^2 + |\hat X|_F^2 - 2 \langle X, \hat X\rangle
$$
and the **relative reconstruction error** stored in `rec_errors` is
$$\frac{|X - \hat X|_F}{|X|_F}$$


In [11]:
start = time.time()
tensor_coo = tensor.tocoo()
norm_tensor = cp.sqrt((cp.abs(tensor_coo.data) ** 2).sum())

norm_X_sq     = norm_tensor ** 2
norm_Xhat_sq  = tl.sum(core * denominator)
inner_prod    = tl.sum(numerator * core)
residual_norm = tl.sqrt(norm_X_sq + norm_Xhat_sq - 2 * inner_prod)
relative_error = residual_norm / norm_tensor
error_time_frob = print_elapsed_time(start, "Reconstruction error done.")

Reconstruction error done.
Elapsed time: 0 minutes and 0.01933884620666504 seconds.


In [8]:
from tucker_tensor import SparseTupleTensor
import tensorly_custom as tl

sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset="fineweb_large",
    method="sc",
    dims=3000,
    map_location="cpu"
)
sparse_tensor.tensor_to_sparse("cupy")
n_iter_max = 10
init = "random"
tol = 10e-5
random_state = 42
verbose = False
return_errors = "full"
normalize_factors = False
patience: int = 3
rank = [100, 100, 100]
shape = tuple(sparse_tensor.shape)
tensor = sparse_tensor.tensor
rank = validate_tucker_rank(shape, rank=rank)
epsilon = 1e-12
modes = list(range(len(rank)))
non_negative = True
# skip_factor = None
# transpose_factors = False


if init == "random":
    rng = tl.check_random_state(random_state)
    core = tl.tensor(
        rng.random_sample([rank[index] for index in range(len(modes))]) + 0.01,
        **tl.context(tensor),
    )
    factors = [
        tl.tensor(
            rng.random_sample((shape[mode], rank[index])),  # we changed this to original shape
            **tl.context(tensor),
        )
        for index, mode in enumerate(modes)
    ]
else:
    (core, factors) = init

if non_negative is True:
    factors = [tl.abs(f) for f in factors]
    core = tl.abs(core)
else:
    raise NotImplementedError("Currently only non-negative=True is supported.")


# Kullback-Leibler divergence

## Factors
Update rule becomes $$
A^{(n)}_{i_n \lambda}
\leftarrow
A^{(n)}_{i_n \lambda},
\frac{
\displaystyle
\sum_{j}
\frac{X_{(n),i_n j}}{\hat{X}_{(n),i_n j}}
B^{(n)}_{j\lambda}
}{
\displaystyle
\sum_{j} B^{(n)}_{j\lambda}
}
$$
Or, for each mode:
$$
A^{(n)} \leftarrow A^{(n)} \odot
\frac{
\left( \frac{X_{(n)}}{\hat{X}_{(n)}} \right) B
}{
\mathbf{1} B
}$$

where $\mathbf{1}$ is a matrix of ones with the same shape as $X_{(n)}$.


Or, exactly following the Morup paper notation:
$$R_{(n)} = A^{(n)}Z_{(n)}$$
$$A^{(n)} \leftarrow A^{(n)}\cdot (\frac{\frac{X_{(n)}}{R_{(n)}}Z^{T}_{(n)}}{E_{(n)}Z^{T}_{(n)}})$$
where $E_{(n)}$ is a matrix of ones with the same shape as $X_{(n)}$


In [10]:
factors_copy = factors.copy()

In [18]:
start = time.time()
for mode_n in modes:
    X = unfold_from_vectorized_sparse(tensor, shape, mode_n)
    Z = tucker_to_tensor((core, factors), skip_factor=mode_n)
    Z = unfold(Z, mode_n) # (50, 1000000)
    A = factors[mode_n] # (1000, 50)


    # this breaks.
    R = tl.dot(A, Z) # (1000, 1000000)
    Z_T = tl.transpose(Z) # (1000000, 50)
    frac = X / R # (1000, 100000)

    numerator = frac @ Z_T # (1000, 50)

    # Denominator: E Z^T  ==  row-wise broadcast of sum over columns of Z
    den_row = tl.sum(Z, axis=1)    # (R,)
    denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)

    # Multiplicative update
    A = A * (numerator / (denominator + 1e-12))

    A = tl.clip(A, a_min=epsilon, a_max=None)

    factors[mode_n] = A
a = print_elapsed_time(start, "KL factors calculated:")

R type: <class 'cupy.ndarray'>, shape (1000, 1000000),
X type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
Z type: <class 'cupy.ndarray'>, shape (50, 1000000),
frac type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
numerator type: <class 'cupy.ndarray'>, shape (1000, 50)
R type: <class 'cupy.ndarray'>, shape (1000, 1000000),
X type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
Z type: <class 'cupy.ndarray'>, shape (50, 1000000),
frac type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
numerator type: <class 'cupy.ndarray'>, shape (1000, 50)
R type: <class 'cupy.ndarray'>, shape (1000, 1000000),
X type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
Z type: <class 'cupy.ndarray'>, shape (50, 1000000),
frac type: <class 'cupyx.scipy.sparse._coo.coo_matrix'>, shape (1000, 1000000),
numerator type: <class 'cupy.ndarray'>, shape (1000, 50)
KL factors calculated:
Elapsed tim

In [11]:
start = time.time()
for mode_n in modes:
    X = unfold_from_vectorized_sparse(tensor, shape, mode_n)
    Z = tucker_to_tensor((core, factors_copy), skip_factor=mode_n)
    Z = unfold(Z, mode_n) # (50, 1000000)
    A = factors_copy[mode_n] # (1000, 50)

    rows = X.row        # (nnz,)
    cols = X.col        # (nnz,)
    vals = X.data       # (nnz,)

    R_nz = cp.zeros_like(vals)  # (nnz,)

    for r in range(rank[0]):          # R = rank[mode_n], e.g. 50
        A_r = A[:, r]           # (I,)
        Z_r = Z[r, :]           # (K,)

        # Gather A[i,r] for all nonzeros and Z[r,j] for all nonzeros:
        A_r_vals = A_r[rows]    # (nnz,)
        Z_r_vals = Z_r[cols]    # (nnz,)

        R_nz += A_r_vals * Z_r_vals  # elementwise, stays (nnz,)
    R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
    w = vals / R_nz            # (nnz,)
    w = w[:, cp.newaxis]       # (nnz, 1)

    W_data = (vals / R_nz)     # (nnz,)
    W = cpx_sparse.coo_matrix(
        (W_data, (rows, cols)),
        shape=X.shape,         # (I_n, K)
    )
    print(f"W shape: {W.shape}, Z shape {Z.shape}")
    numerator = W @ tl.transpose(Z)          # (I_n, R)
    # # this breaks.
    # R = tl.dot(A, Z) # (1000, 1000000)
    # Z_T = tl.transpose(Z) # (1000000, 50)
    # frac = X / R # (1000, 100000)
    # numerator = frac @ Z_T # (1000, 50)

    # Denominator: E Z^T  ==  row-wise broadcast of sum over columns of Z
    den_row = tl.sum(Z, axis=1)    # (R,)
    denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)

    # Multiplicative update
    A = A * (numerator / (denominator + 1e-12))

    A = tl.clip(A, a_min=epsilon, a_max=None)

    factors_copy[mode_n] = A
a = print_elapsed_time(start, "KL factors calculated:")

W shape: (3000, 9000000), Z shape (100, 9000000)
W shape: (3000, 9000000), Z shape (100, 9000000)
W shape: (3000, 9000000), Z shape (100, 9000000)
KL factors calculated:
Elapsed time: 0 minutes and 0.533268928527832 seconds.


In [27]:
# we check if core and core_copy are exactly the same (up to numerical tolerance)
atol = 1e-6
rtol = 1e-6

diff_found = False
for i, (slc_core, slc_copy) in enumerate(zip(factors, factors_copy)):
    if not np.allclose(slc_core, slc_copy, atol=atol, rtol=rtol):
        print(f"Mismatch in slice {i}")
        max_diff = tl.max(tl.abs(slc_core - slc_copy))
        print(f"Max abs diff in this slice: {max_diff}")
        # we print the mean of the slice
        mean = tl.mean(slc_core)
        mean_copy = tl.mean(slc_copy)
        print(f"Mean in the slices: {mean_copy} - {mean}")
        diff_found = True


if not diff_found:
    print("core and core_copy match within numerical tolerance.")

Mismatch in slice 1
Max abs diff in this slice: 8.20159912109375e-05
Mean in the slices: 0.5003954172134399 - 0.5003954172134399
Mismatch in slice 2
Max abs diff in this slice: 7.2479248046875e-05
Mean in the slices: 0.5018934607505798 - 0.5018934607505798


# Core
Following Morup 2008
$$
\mathcal{G}_\lambda \leftarrow
\mathcal{G}_\lambda \odot
\frac{
\displaystyle \sum_i \Phi_{i,\lambda} , \frac{X_i}{\hat{X}_i}
}{
\displaystyle \sum_i \Phi_{i,\lambda}
}
$$
with $$
\Phi_{i, \lambda} = \prod_{n=1}^N A^{(n)}_{i_n \lambda_n}.
$$

In [12]:
import numpy as np
import cupy as cp
import cupyx.scipy.sparse as cpx_sparse
import tensorly_custom as tl
from tensorly_custom.decomposition._tucker import tucker_to_tensor

def build_sparse_ratio_vec(
    vec_tensor: cpx_sparse.spmatrix,
    shape,
    core,
    factors,
    epsilon: float = 1e-12,
) -> cpx_sparse.coo_matrix:
    """Build a sparse vectorized tensor Y with the same structure as X,
    but data = X_i / R_i, where R is the current reconstruction.

    vec_tensor : vectorized sparse X (block-encoded)
    shape      : original N-D shape
    core       : current core G
    factors    : list of A^{(n)}
    """
    shape = tuple(shape)
    size = int(np.prod(shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    # 1) Dense reconstruction R = G ×_1 A^{(1)} × ... ×_N A^{(N)}
    #    (follows the paper literally; for huge tensors this may be too big
    #     and you’d want a custom "R at nonzeros only" kernel instead.)
    R = tucker_to_tensor((core, factors))        # dense cp.ndarray of shape `shape`
    R = tl.clip(R, a_min=epsilon, a_max=None)
    R_flat = R.ravel()                           # length = size

    # 2) Same block encoding as elsewhere
    coo = vec_tensor.tocoo()
    row_cp = coo.row
    col_cp = coo.col
    data_cp = coo.data

    # move indices to host to do int64 arithmetic safely
    row_np = cp.asnumpy(row_cp).astype(np.int64)
    col_np = cp.asnumpy(col_cp).astype(np.int64)

    flat_np = row_np + col_np * np.int64(block_size)  # flat indices into R_flat
    flat_cp = cp.asarray(flat_np)

    # 3) Compute X_i / R_i at the same positions
    R_vals = R_flat[flat_cp]                          # cp.ndarray
    R_vals = tl.clip(R_vals, a_min=epsilon, a_max=None)

    ratio_data = data_cp / R_vals                     # X_i / R_i

    # 4) Build sparse ratio vec with identical row, col, shape
    vec_ratio = cpx_sparse.coo_matrix(
        (ratio_data, (row_cp, col_cp)),
        shape=vec_tensor.shape,
    )
    #vec_ratio.sum_duplicates()  # just in case

    return vec_ratio


In [13]:
def kl_core_update(
    tensor,        # vectorized sparse X
    shape,         # original N-D shape
    core,          # current core G
    factors,       # list of factor matrices A^{(n)}
    epsilon=1e-12,
):
    """KL core update following Mørup et al. (2008):

       G <- G ⊙ (D / F),

       D = (X / R) ×_1 A^{(1)T} × ... ×_N A^{(N)T}
       F = E ×_1 A^{(1)T} × ... ×_N A^{(N)T},
           with E all-ones, so F is an outer product of column sums.
    """
    shape = tuple(shape)
    modes = list(range(len(shape)))

    # --- 1) Build sparse ratio Y = X / R with same structure as X ---
    vec_ratio = build_sparse_ratio_vec(
        vec_tensor=tensor,
        shape=shape,
        core=core,
        factors=factors,
        epsilon=epsilon,
    )

    # --- 2) D = (X/R) ×_1 A^{(1)T} × ... ×_N A^{(N)T} ---
    D = sparse_multi_mode_dot_vec(
        vec_tensor=vec_ratio,
        orig_shape=shape,
        factors=factors,
        modes=modes,
        transpose_factors=True,  # uses A^{(n)T}, consistent with Tucker
    )
    D = tl.clip(D, a_min=epsilon, a_max=None)

    # --- 3) F = E ×_1 A^{(1)T} × ... ×_N A^{(N)T} ---
    # Since E ≡ 1, F is the N-way outer product of column sums
    col_sums = [tl.sum(A_n, axis=0) for A_n in factors]  # s^{(n)} ∈ R^{R_n}

    # Start from first mode, shape (R_1, 1, 1, ..., 1)
    F = col_sums[0].reshape((core.shape[0],) + (1,) * (core.ndim - 1))

    for n in range(1, core.ndim):
        # reshape s^{(n)} to broadcast along λ_n
        # e.g. for n=1 in 3D core: (1, R_2, 1)
        shape_n = [1] * n + [core.shape[n]] + [1] * (core.ndim - n - 1)
        F = F * col_sums[n].reshape(tuple(shape_n))

    F = tl.clip(F, a_min=epsilon, a_max=None)

    # --- 4) Multiplicative update G <- G ⊙ D/F (α = 1) ---
    core *= D / F

    return core


In [14]:
# After KL factor updates:
start = time.time()
core = kl_core_update(
    tensor=tensor,
    shape=shape,
    core=core,
    factors=factors,
    epsilon=epsilon,
)
core_time_kl = print_elapsed_time(start, "Core update done.")


OutOfMemoryError: Out of memory allocating 108,000,000,000 bytes (allocated so far: 11,031,126,528 bytes).

$\mathcal{R} = \mathcal{G} \Phi$

$\mathcal{D} = \frac{\mathcal{X}}{\mathcal{R}} \Phi^T$

$\mathcal{F} = \mathcal{E} \Phi^T$

$$\mathcal{G} \leftarrow \mathcal{G} \cdot (\frac{\mathcal{D}}{\mathcal{F}})$$

In [19]:
start = time.time()
R = tucker_to_tensor((core_copy, factors))
# we flatten to a sparse tensor, with values only where T has them
R = R.ravel().reshape(tensor.shape, order="F")
print(R.shape)
R = tl.clip(R, a_min=epsilon, a_max=None)
print(tensor.shape)
X_R = tensor / R
print(X_R.shape)
print(type(X_R))
# for i, factor in enumerate(factors):
#     print(factor.shape)
#     X_R = mode_dot(X_R, tl.transpose(factor), i)
X_R = sparse_multi_mode_dot_vec(
    vec_tensor=X_R,
    orig_shape=shape,
    factors=factors,
    modes=modes,
    transpose_factors=True,  # X ×_n W_n^T
)
X_R = tl.clip(X_R, a_min=epsilon, a_max=None)

col_sums = [tl.sum(A_n, axis=0) for A_n in factors]
F = col_sums[0].reshape((core_copy.shape[0],) + (1,) * (core_copy.ndim - 1))
for n in range(1, core_copy.ndim):
    shape_n = [1] * n + [core_copy.shape[n]] + [1] * (core_copy.ndim - n - 1)
    F = F * col_sums[n].reshape(tuple(shape_n))
F = tl.clip(F, a_min=epsilon, a_max=None)
core_copy *= X_R / F
core_time_kl = print_elapsed_time(start, "Core update done.")

NameError: name 'core_copy' is not defined

In [38]:
iters = 50
core_copy = core.copy()
factors_copy = factors.copy()
for i in range(iters):
    mean_f = [f.mean() for f in factors_copy]
    core_mean = core_copy.mean()
    print(f"Means at iteration {i}: core: {core_mean}\n factors: {mean_f}")
    for mode in modes:
        X = unfold_from_vectorized_sparse(tensor, shape, mode)
        Z = tucker_to_tensor((core, factors_copy), skip_factor=mode)
        Z = unfold(Z, mode) # (50, 1000000)
        A = factors_copy[mode] # (1000, 50)

        rows = X.row        # (nnz,)
        cols = X.col        # (nnz,)
        vals = X.data       # (nnz,)

        R_nz = cp.zeros_like(vals)  # (nnz,)

        for r in range(rank[0]):          # R = rank[mode], e.g. 50
            A_r = A[:, r]           # (I,)
            Z_r = Z[r, :]           # (K,)

            # Gather A[i,r] for all nonzeros and Z[r,j] for all nonzeros:
            A_r_vals = A_r[rows]    # (nnz,)
            Z_r_vals = Z_r[cols]    # (nnz,)

            R_nz += A_r_vals * Z_r_vals  # elementwise, stays (nnz,)
        R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)
        w = vals / R_nz            # (nnz,)
        w = w[:, cp.newaxis]       # (nnz, 1)

        W_data = (vals / R_nz)     # (nnz,)
        W = cpx_sparse.coo_matrix(
            (W_data, (rows, cols)),
            shape=X.shape,         # (I_n, K)
        )
        # print(f"W shape: {W.shape}, Z shape {Z.shape}")
        numerator = W @ tl.transpose(Z)          # (I_n, R)
        # # this breaks.
        # R = tl.dot(A, Z) # (1000, 1000000)
        # Z_T = tl.transpose(Z) # (1000000, 50)
        # frac = X / R # (1000, 100000)
        # numerator = frac @ Z_T # (1000, 50)

        # Denominator: E Z^T  ==  row-wise broadcast of sum over columns of Z
        den_row = tl.sum(Z, axis=1)    # (R,)
        denominator = den_row[np.newaxis, :]   # (1, R) – will broadcast to (I, R)

        # Multiplicative update
        A = A * (numerator / (denominator + 1e-12))

        A = tl.clip(A, a_min=epsilon, a_max=None)

        factors_copy[mode] = A



    R = tucker_to_tensor((core_copy, factors_copy))

    # we flatten to a sparse tensor, with values only where T has them
    R = R.ravel().reshape(tensor.shape, order="F")
    R = tl.clip(R, a_min=epsilon, a_max=None)
    X_R = tensor / R

    X_R = sparse_multi_mode_dot_vec(
        vec_tensor=X_R,
        orig_shape=shape,
        factors=factors_copy,
        modes=modes,
        transpose_factors=True,  # X ×_n W_n^T
    )
    X_R = tl.clip(X_R, a_min=epsilon, a_max=None)

    col_sums = [tl.sum(A_n, axis=0) for A_n in factors_copy]
    F = col_sums[0].reshape((core_copy.shape[0],) + (1,) * (core_copy.ndim - 1))
    for n in range(1, core_copy.ndim):
        shape_n = [1] * n + [core_copy.shape[n]] + [1] * (core_copy.ndim - n - 1)
        F = F * col_sums[n].reshape(tuple(shape_n))
    F = tl.clip(F, a_min=epsilon, a_max=None)
    core_copy *= X_R / (F + epsilon)

    core_copy, factors_copy = tucker_normalize((core_copy, factors_copy))



Means at iteration 0: core: 0.5092774629592896
 factors: [array(0.50269735, dtype=float32), array(0.50044763, dtype=float32), array(0.5018805, dtype=float32)]
Means at iteration 1: core: 0.01057345699518919
 factors: [array(0.01204289, dtype=float32), array(0.01003437, dtype=float32), array(0.01574537, dtype=float32)]
Means at iteration 2: core: 0.011554955504834652
 factors: [array(0.01190021, dtype=float32), array(0.00982917, dtype=float32), array(0.0150584, dtype=float32)]
Means at iteration 3: core: 0.01164172776043415
 factors: [array(0.0118897, dtype=float32), array(0.00982411, dtype=float32), array(0.01496951, dtype=float32)]
Means at iteration 4: core: 0.0116371288895607
 factors: [array(0.01188853, dtype=float32), array(0.00982387, dtype=float32), array(0.01495783, dtype=float32)]
Means at iteration 5: core: 0.011621587909758091
 factors: [array(0.01188838, dtype=float32), array(0.00982382, dtype=float32), array(0.01495629, dtype=float32)]
Means at iteration 6: core: 0.0116050

In [93]:
# we count the nonzero values in the core and in each of the factors
nnz_core = tl.sum(core > 0).item()
nnz_factors = [tl.sum(f > 0).item() for f in factors]
print(f"Number of nonzeros in core: {nnz_core}")
for i, nnz in enumerate(nnz_factors):
    print(f"Number of nonzeros in factor {i}: {nnz}")
negative_core = tl.sum(core < 0).item()
negative_factors = [tl.sum(f < 0).item() for f in factors]
print(f"Number of negative nonzeros in core: {negative_core}")
for i, nnz in enumerate(negative_factors):
    print(f"Number of negative nonzeros in factor {i}: {nnz}")

Number of nonzeros in core: 125000
Number of nonzeros in factor 0: 49900
Number of nonzeros in factor 1: 50000
Number of nonzeros in factor 2: 50000
Number of negative nonzeros in core: 0
Number of negative nonzeros in factor 0: 100
Number of negative nonzeros in factor 1: 0
Number of negative nonzeros in factor 2: 0


# Reconstruction error
As Morup:
$$C_{\text{KL}}(X, R) = \sum_i X_i \log\frac{X_i}{R_i} - X_i + R_i$$

In [22]:
# X = original tensor
# R = reconstructed from core G and factors A
R = tucker_to_tensor((core, factors))
print("shapes")
print(f"original shape: {tensor.shape}")
print(f"R shape: {R.shape}")
# we flatten to a (1000000000, 1) shape
R = R.ravel()
print(R.shape)
cp.sum(tensor * cp.log(tensor / R) - tensor + R)


Number of nonzeros in core: 1000000000, negatives: 0
Number of nonzeros in R_flat: 1000000000, negatives: 0


In [23]:
shape = tuple(shape)
size = int(np.prod(shape))
int32_max  = np.iinfo(np.int32).max
block_size = min(size, int32_max)

# --- 1) Dense reconstruction R = G ×_1 A^{(1)} × ... ×_N A^{(N)} ---
# This is exactly step 3 in Table 2 of the paper.
reconstruction = tucker_to_tensor((core, factors))      # cp.ndarray, shape=shape
nnz_R = tl.sum(reconstruction > 0).item()
negative_R = tl.sum(reconstruction < 0).item()
print(f"Number of nonzeros in core: {nnz_R}, negatives: {negative_R}")
reconstruction = tl.clip(reconstruction, a_min=epsilon, a_max=None)
R_flat = reconstruction.ravel()                         # length = size
nnz_R = tl.sum(R_flat > 0).item()
negative_R = tl.sum(R_flat < 0).item()
print(f"Number of nonzeros in R_flat: {nnz_R}, negatives: {negative_R}")

(754107,)
Number of nonzeros in flat_cp: 754106, negatives: 0


In [24]:
# --- 2) Decode sparse X indices to flat indices ---
X_coo  = tensor.tocoo()
row_cp = X_coo.row
col_cp = X_coo.col
x_data = X_coo.data

# same block-encoding as in unfold_from_vectorized_sparse
row_np = cp.asnumpy(row_cp).astype(np.int64)
col_np = cp.asnumpy(col_cp).astype(np.int64)
flat_np = row_np + col_np * np.int64(block_size)
flat_cp = cp.asarray(flat_np)
# we print the shape of flat_cp
print(flat_cp.shape)
nnz_R = tl.sum(flat_cp > 0).item()
negative_R = tl.sum(flat_cp < 0).item()
print(f"Number of nonzeros in flat_cp: {nnz_R}, negatives: {negative_R}")

Number of nonzeros in r_nz: 754107, negatives: 0
Number of nonzeros in r_nz: 754107, negatives: 0
Number of nonzeros in x_nz: 653363, negatives: 100744
Number of nonzeros in x_nz: 754107, negatives: 0
(754107,) (754107,)


In [25]:
# --- 3) X_i and R_i at nonzero entries ---
r_nz = R_flat[flat_cp]
nnz_R = tl.sum(r_nz > 0).item()
negative_R = tl.sum(r_nz < 0).item()
print(f"Number of nonzeros in r_nz: {nnz_R}, negatives: {negative_R}")
# R_i where X_i ≠ 0
r_nz = tl.clip(r_nz, a_min=epsilon, a_max=None)
nnz_R = tl.sum(r_nz > 0).item()
negative_R = tl.sum(r_nz < 0).item()
print(f"Number of nonzeros in r_nz: {nnz_R}, negatives: {negative_R}")
x_nz = x_data
nnz_R = tl.sum(x_nz > 0).item()
negative_R = tl.sum(x_nz < 0).item()
print(f"Number of nonzeros in x_nz: {nnz_R}, negatives: {negative_R}")
x_nz = tl.clip(x_nz, a_min=epsilon, a_max=None)
nnz_R = tl.sum(x_nz > 0).item()
negative_R = tl.sum(x_nz < 0).item()
print(f"Number of nonzeros in x_nz: {nnz_R}, negatives: {negative_R}")
print(r_nz.shape, x_nz.shape)

(754107,)
11247312.0


In [26]:
# --- 4) KL contribution from nonzeros ---
# sum_{i: X_i>0} [X_i log(X_i/R_i) - X_i + R_i]
term_pos = x_nz * cp.log(x_nz / r_nz) - x_nz + r_nz
print(term_pos.shape)
kl_pos = cp.sum(term_pos)
print(kl_pos)

In [27]:
# --- 5) KL contribution from zero entries ---
# For X_i = 0, the KL term tends to R_i (limit X→0).
# Full sum over all i is:
#   ∑_i [X_i log(X_i/R_i) - X_i + R_i]
# = (sum over nonzeros) + (sum over zeros),
# and sum_{zeros} R_i = sum(R) - sum_{nonzeros} R_i.
sum_R      = cp.sum(R)
sum_R_nz   = cp.sum(r_nz)
kl_zero    = sum_R - sum_R_nz
kl_total   = kl_pos + kl_zero


KL divergence: 13398802.0, relative KL: 5.514830589294434


In [ ]:
# --- 6) Optional normalized "relative" KL error ---
sum_X = cp.sum(x_nz)  # sum over nonzero X
rel_kl = kl_total / cp.maximum(sum_X, epsilon)
print(f"KL divergence: {kl_total}, relative KL: {rel_kl}")
